# Interpretação dos Resultados — Rain in Australia

**PCO213 — Aprendizado de Máquina e Mineração de Dados | UNIFEI**

Este notebook complementa o `02_modeling.ipynb` com análise qualitativa dos resultados.
Recarrega os artefatos exportados (`models/best_model.joblib` e tabelas de `outputs/tables/`)
para manter consistência com o que foi treinado.

**Seções:**
1. Carregamento dos artefatos
2. Análise da tabela de métricas
3. Trade-off Precision × Recall
4. Interpretação das variáveis mais importantes
5. Discussão e limitações
6. Sumário para o relatório

In [ ]:
import sys
from pathlib import Path

def _find_project_root() -> Path:
    for candidate in [Path('.'), Path('..')]:
        resolved = candidate.resolve()
        if (resolved / 'src').exists() and (resolved / 'requirements.txt').exists():
            return resolved
    raise FileNotFoundError("Raiz do projeto nao encontrada.")

_root = _find_project_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', '{:.4f}'.format)
print(f"Raiz do projeto: {_root}")

---
## Seção 1 — Carregamento dos Artefatos

> **Pré-requisito:** execute `02_modeling.ipynb` primeiro (ou `python main.py`).
> Este notebook lê arquivos gerados pelo pipeline de modelagem.

In [ ]:
import joblib
from src.config import MODELS_DIR, TABLES_DIR, FIGURES_DIR

# Carrega melhor modelo
model_path = MODELS_DIR / 'best_model.joblib'
if model_path.exists():
    artifact = joblib.load(model_path)
    best_name     = artifact['model_name']
    best_pipeline = artifact['pipeline']
    print(f"Melhor modelo carregado: {best_name}")
else:
    print(f"AVISO: {model_path} nao encontrado. Execute 02_modeling.ipynb primeiro.")
    best_name, best_pipeline = None, None

# Carrega tabela de métricas
metrics_path = TABLES_DIR / 'metrics_comparison.csv'
if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path)
    print(f"Tabela de metricas carregada: {metrics_path.name}")
else:
    print(f"AVISO: {metrics_path} nao encontrado.")
    df_metrics = None

---
## Seção 2 — Análise da Tabela de Métricas

Comparação entre modelos com destaque para F1 e Recall.
O Recall é prioritário pois falsos negativos (prever "não chove" quando chove)
têm custo prático maior em planejamento agrícola, defesa civil e logística.

In [ ]:
if df_metrics is not None:
    # Destaca o melhor modelo por F1
    display(df_metrics.style
        .highlight_max(subset=['accuracy','precision','recall','f1','roc_auc'], color='#d4edda')
        .highlight_min(subset=['accuracy','precision','recall','f1','roc_auc'], color='#f8d7da')
        .format({
            'accuracy': '{:.4f}', 'precision': '{:.4f}',
            'recall': '{:.4f}', 'f1': '{:.4f}',
            'roc_auc': '{:.4f}', 'avg_precision': '{:.4f}'
        })
    )

---
## Seção 3 — Trade-off Precision × Recall

Em um problema de previsão de chuva:
- **Falso Negativo (FN):** predizer "não chove" quando chove → impacto em agricultura, defesa civil, eventos ao ar livre.
- **Falso Positivo (FP):** predizer "chove" quando não chove → causa precaução desnecessária (custo menor).

Por isso, **Recall** (sensibilidade) é a métrica de maior interesse prático neste contexto.

In [ ]:
if df_metrics is not None:
    df_tradeoff = df_metrics[['modelo', 'precision', 'recall', 'f1']].copy()
    df_tradeoff['diferenca_recall_precision'] = (df_tradeoff['recall'] - df_tradeoff['precision']).round(4)
    df_tradeoff = df_tradeoff.sort_values('recall', ascending=False)
    print("Trade-off Recall x Precision por modelo:")
    print(df_tradeoff.to_string(index=False))
    print()
    print("Valores positivos em 'diferenca_recall_precision' indicam modelos")
    print("que priorizam Recall sobre Precision (desejavel neste contexto).")

---
## Seção 4 — Figuras Geradas

Exibe as figuras de avaliação geradas pelo pipeline para referência no artigo.

In [ ]:
from IPython.display import Image, display as ipy_display

fig_files = [
    ('07_confusion_matrix.png',   'Fig 7 — Matriz de Confusao do Melhor Modelo'),
    ('08_roc_curves.png',         'Fig 8 — Curvas ROC de Todos os Modelos'),
    ('09_pr_curve.png',           'Fig 9a — Curva Precision-Recall'),
    ('09_metrics_comparison.png', 'Fig 9b — Comparativo de Metricas'),
    ('10_feature_importance.png', 'Fig 10 — Importancia das Variaveis'),
]

for fname, caption in fig_files:
    fpath = FIGURES_DIR / fname
    if fpath.exists():
        print(f"\n{caption}")
        ipy_display(Image(filename=str(fpath), width=900))
    else:
        print(f"  [AVISO] {fname} nao encontrado. Execute 02_modeling.ipynb primeiro.")

---
## Seção 5 — Discussão: Variáveis Mais Importantes

Com base na Fig 10 (importância de variáveis), analise quais atributos
meteorológicos mais contribuíram para a predição.

**Hipóteses esperadas** (a confirmar com os valores reais após execução):
- `Humidity3pm` — umidade relativa à tarde; alta correlação com chuva no dia seguinte.
- `Pressure3pm` — queda de pressão à tarde indica frentes de baixa pressão.
- `Rainfall` — chuva acumulada no dia atual (relacionada a `RainToday`).
- `Cloud3pm` — nebulosidade à tarde.
- `WindGustSpeed` — rajadas fortes associadas a sistemas frontais.

> **Nota:** os valores de importância e as posições no ranking são gerados
> após execução real do modelo. Nenhum número é inventado aqui.

In [ ]:
import numpy as np

if best_pipeline is not None and best_name == 'RandomForest':
    try:
        preprocessor = best_pipeline.named_steps['preprocessor']
        feature_names = list(preprocessor.get_feature_names_out())
        importances   = best_pipeline.named_steps['classifier'].feature_importances_

        df_imp = pd.DataFrame({
            'feature':    feature_names,
            'importance': importances,
        }).sort_values('importance', ascending=False).head(20)
        df_imp['importance'] = df_imp['importance'].round(4)

        print(f"Top 20 features — {best_name}:")
        print(df_imp.to_string(index=False))
    except Exception as e:
        print(f"Erro ao obter importancias: {e}")
else:
    print(f"Melhor modelo eh {best_name}. Importancias disponiveis apenas para RandomForest.")
    print("Consulte a Fig 10 para ver os coeficientes da Logistic Regression.")

---
## Seção 6 — Sumário para o Relatório

Células de sumário para copiar/adaptar no artigo científico.

> **Preencha os valores em colchetes com os resultados reais após execução.**

In [ ]:
if df_metrics is not None and best_name is not None:
    best_row = df_metrics[df_metrics['modelo'] == best_name].iloc[0]
    baseline = df_metrics[df_metrics['modelo'] == 'Baseline'].iloc[0]

    print("=" * 60)
    print("SUMARIO PARA O RELATORIO")
    print("=" * 60)
    print(f"""
Melhor modelo: {best_name}
  Accuracy  : {best_row['accuracy']}
  Precision : {best_row['precision']}
  Recall    : {best_row['recall']}
  F1-score  : {best_row['f1']}
  ROC-AUC   : {best_row['roc_auc']}

Baseline (DummyClassifier - majoritario):
  F1        : {baseline['f1']}
  ROC-AUC   : {baseline['roc_auc']}

Ganho F1 vs. Baseline : {float(best_row['f1']) - float(baseline['f1']):.4f}
Ganho ROC-AUC vs. Base: {float(best_row['roc_auc']) - float(baseline['roc_auc']):.4f}
""")
else:
    print("Execute 02_modeling.ipynb para gerar os resultados.")

---
## Seção 7 — Limitações e Trabalhos Futuros

**Limitações do dataset:**
- Dados exclusivos da Austrália: modelos não generalizáveis diretamente a outras regiões.
- Alto percentual de ausentes em colunas meteorológicas relevantes (ex.: Sunshine, Evaporation).
- Dados históricos sem garantia de atualização; condições climáticas mudam com o tempo.

**Limitações dos modelos:**
- Modelos tratam cada dia como observação independente (sem informação sequencial/temporal).
- Imputação por mediana pode introduzir viés em colunas com muitos ausentes.
- `class_weight='balanced'` pode sacrificar Precision para ganhar Recall.

**Trabalhos futuros:**
- Modelos de séries temporais (LSTM, Prophet) para capturar dependência temporal.
- Testar XGBoost / LightGBM / CatBoost.
- Avaliação por localidade (estação meteorológica).
- Calibração probabilística (Platt Scaling, Isotônica).
- API REST ou dashboard interativo (Streamlit/Gradio) para visualização em tempo real.
- Uso de dados meteorológicos mais recentes via API do Bureau of Meteorology.